In [1]:
"""
02_data_preprocessing.ipynb

Purpose:
Transform raw Telstra-style relational tables into model-ready tabular datasets
for train and test, without target leakage.

What this notebook does:
1. Load raw datasets
2. Convert text labels into numeric / standardized feature names
3. Build wide sparse matrices from relational tables
4. Create unsupervised engineered features
5. Merge everything into final processed train and test tables
6. Save only the important outputs needed for later notebooks

Outputs saved to:
    ../data/processed/
    ../reports/preprocessing/
"""

# 1. Imports and Environment Setup

# sys is used only to print Python version
import sys

# Path helps build file paths cleanly
from pathlib import Path

# numpy is used for numerical operations
import numpy as np

# pandas is used for loading, transforming, and saving tabular data
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

# Show more columns when printing dataframes
pd.set_option("display.max_columns", None)

# Make dataframe output wider in notebook
pd.set_option("display.width", 180)

# Fix random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Print versions
print("Python :", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


# 2. Paths

# Assume notebook runs from /notebooks
PROJECT_ROOT = Path.cwd().parent

# Folder with raw CSV files
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"

# Folder where processed train/test tables will be saved
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"

# Folder for small report files
REPORTS_PATH = PROJECT_ROOT / "reports" / "preprocessing"

# Create folders if they do not already exist
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT   :", PROJECT_ROOT)
print("RAW_DATA_PATH  :", RAW_DATA_PATH)
print("PROCESSED_PATH :", PROCESSED_PATH)
print("REPORTS_PATH   :", REPORTS_PATH)


# 3. Helper Functions

def show_df(df: pd.DataFrame, n: int = 5) -> None:
    """
    Show first few rows of a dataframe.
    """
    if display is not None:
        display(df.head(n))
    else:
        print(df.head(n))


def extract_num(series: pd.Series, col_name: str = "") -> pd.Series:
    """
    Extract the trailing number from labels like:

    'location 118'     -> 118
    'event_type 35'    -> 35
    'resource_type 8'  -> 8
    'severity_type 2'  -> 2
    'log_feature 68'   -> 68

    This is useful because the raw data stores many values as text labels.
    Later, the model needs numeric-friendly features.
    """
    # If already numeric, just convert to int
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int)

    # Convert values to string and remove extra spaces
    s = series.astype(str).str.strip()

    # Extract number at the end of the string
    out = s.str.extract(r"(\d+)$")[0]

    # Stop if extraction fails for any value
    if out.isna().any():
        bad = s[out.isna()].unique()[:5]
        raise ValueError(f"Failed numeric extraction in {col_name}. Examples: {bad}")

    return out.astype(int)


def add_prefix_from_numeric(series: pd.Series, prefix: str) -> pd.Series:
    """
    Convert label values into standardized feature names.

    Example:
    event_type 35 -> event_type_35
    resource_type 2 -> resource_type_2
    """
    return prefix + extract_num(series).astype(str)


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    """
    Save dataframe as CSV.
    """
    df.to_csv(path, index=index)


def detect_group(col: str) -> str:
    """
    Classify a feature into its group.

    This is used only for the feature inventory report.
    """
    if col == "location_num":
        return "location"
    if col.startswith("severity_type_"):
        return "severity_type"
    if col.startswith("event_type_"):
        return "event_type"
    if col.startswith("resource_type_"):
        return "resource_type"
    if col.startswith("log_feature_") and col.endswith("_vol"):
        return "log_feature_volume"
    if col.startswith("log_feature_"):
        return "log_feature_presence"
    return "engineered"


# 4. Load Raw Datasets

# Load all raw tables
train = pd.read_csv(RAW_DATA_PATH / "train.csv")
test = pd.read_csv(RAW_DATA_PATH / "test.csv")
event_type = pd.read_csv(RAW_DATA_PATH / "event_type.csv")
log_feature = pd.read_csv(RAW_DATA_PATH / "log_feature.csv")
resource_type = pd.read_csv(RAW_DATA_PATH / "resource_type.csv")
severity_type = pd.read_csv(RAW_DATA_PATH / "severity_type.csv")

print("\nLoaded datasets:")
print("train        :", train.shape)
print("test         :", test.shape)
print("event_type   :", event_type.shape)
print("log_feature  :", log_feature.shape)
print("resource_type:", resource_type.shape)
print("severity_type:", severity_type.shape)


# 5. Standardize Label Columns

print("\nSTANDARDIZING LABELS")

# Work on copies so raw tables remain unchanged
train_proc = train.copy()
test_proc = test.copy()
event_proc = event_type.copy()
log_proc = log_feature.copy()
resource_proc = resource_type.copy()
severity_proc = severity_type.copy()

# Convert location text like "location 344" into a numeric version
train_proc["location_num"] = extract_num(train_proc["location"], "train.location")
test_proc["location_num"] = extract_num(test_proc["location"], "test.location")

# Convert relational label columns into consistent feature names
# Example: event_type 35 becomes event_type_35
event_proc["event_feature"] = add_prefix_from_numeric(event_proc["event_type"], "event_type_")
resource_proc["resource_feature"] = add_prefix_from_numeric(resource_proc["resource_type"], "resource_type_")
severity_proc["severity_feature"] = add_prefix_from_numeric(severity_proc["severity_type"], "severity_type_")
log_proc["log_feature_std"] = add_prefix_from_numeric(log_proc["log_feature"], "log_feature_")

print("Unique locations      :", pd.concat([train_proc["location_num"], test_proc["location_num"]]).nunique())
print("Unique event features :", event_proc["event_feature"].nunique())
print("Unique resource feats :", resource_proc["resource_feature"].nunique())
print("Unique severity feats :", severity_proc["severity_feature"].nunique())
print("Unique log features   :", log_proc["log_feature_std"].nunique())


# 6. Build Base ID Frames

print("\nBUILDING BASE TABLES")

# Keep the important identifying columns from train and test
train_base = train_proc[["id", "location", "location_num", "fault_severity"]].copy()
test_base = test_proc[["id", "location", "location_num"]].copy()

# Combine train and test IDs so we can build one full feature table first
all_ids = pd.concat(
    [
        train_base[["id", "location", "location_num"]],
        test_base[["id", "location", "location_num"]],
    ],
    axis=0,
    ignore_index=True,
).drop_duplicates(subset=["id"]).reset_index(drop=True)

print("Base all_ids shape:", all_ids.shape)
show_df(all_ids)


# 7. Build Sparse Feature Matrices

print("\nBUILDING SPARSE MATRICES")

# Event matrix:
# one column per event_type_xx, values show whether that id has that event
event_matrix = (
    pd.crosstab(event_proc["id"], event_proc["event_feature"])
    .astype(np.uint8)
    .reset_index()
)

# Resource matrix:
# one column per resource_type_xx
resource_matrix = (
    pd.crosstab(resource_proc["id"], resource_proc["resource_feature"])
    .astype(np.uint8)
    .reset_index()
)

# Severity matrix:
# one column per severity_type_xx
severity_matrix = (
    pd.crosstab(severity_proc["id"], severity_proc["severity_feature"])
    .astype(np.uint8)
    .reset_index()
)

# Log presence matrix:
# counts how many times each log feature appears per id
log_presence_matrix = (
    pd.crosstab(log_proc["id"], log_proc["log_feature_std"])
    .astype(np.uint16)
    .reset_index()
)

# Log volume matrix:
# sum of log volume for each log feature per id
log_volume_matrix = (
    log_proc.pivot_table(
        index="id",
        columns="log_feature_std",
        values="volume",
        aggfunc="sum",
        fill_value=0,
    )
    .astype(np.float32)
    .reset_index()
)

# Rename log volume columns so they do not clash with presence columns
log_volume_matrix.columns = [
    col if col == "id" else f"{col}_vol"
    for col in log_volume_matrix.columns
]

print("event_matrix shape       :", event_matrix.shape)
print("resource_matrix shape    :", resource_matrix.shape)
print("severity_matrix shape    :", severity_matrix.shape)
print("log_presence_matrix shape:", log_presence_matrix.shape)
print("log_volume_matrix shape  :", log_volume_matrix.shape)


# 8. Create Engineered Aggregate Features

print("\nCREATING ENGINEERED FEATURES")

# Aggregate basic counts from event table
event_agg = (
    event_proc.groupby("id")
    .agg(
        event_count=("event_feature", "count"),
        unique_event_types=("event_feature", "nunique"),
    )
    .reset_index()
)

# Aggregate basic counts from resource table
resource_agg = (
    resource_proc.groupby("id")
    .agg(
        resource_count=("resource_feature", "count"),
        unique_resource_types=("resource_feature", "nunique"),
    )
    .reset_index()
)

# Aggregate severity information
severity_agg = (
    severity_proc.groupby("id")
    .agg(
        severity_type_count=("severity_feature", "count"),
        unique_severity_types=("severity_feature", "nunique"),
    )
    .reset_index()
)

# Aggregate log statistics
log_agg = (
    log_proc.groupby("id")
    .agg(
        log_count=("log_feature_std", "count"),
        unique_log_features=("log_feature_std", "nunique"),
        log_volume_sum=("volume", "sum"),
        log_volume_mean=("volume", "mean"),
        log_volume_max=("volume", "max"),
        log_volume_std=("volume", "std"),
    )
    .reset_index()
)

# If only one log record exists for an id, std becomes NaN, so replace with 0
log_agg["log_volume_std"] = log_agg["log_volume_std"].fillna(0.0)

# Merge all aggregate features together
agg_features = (
    all_ids[["id", "location_num"]]
    .merge(event_agg, on="id", how="left")
    .merge(resource_agg, on="id", how="left")
    .merge(severity_agg, on="id", how="left")
    .merge(log_agg, on="id", how="left")
)

# These columns should behave like numeric counts/statistics
count_like_cols = [
    "event_count",
    "unique_event_types",
    "resource_count",
    "unique_resource_types",
    "severity_type_count",
    "unique_severity_types",
    "log_count",
    "unique_log_features",
    "log_volume_sum",
    "log_volume_mean",
    "log_volume_max",
    "log_volume_std",
]

# Fill missing aggregate values with 0
for col in count_like_cols:
    agg_features[col] = agg_features[col].fillna(0)

# Ratio features
# These help describe behavior patterns without using the target
agg_features["events_per_resource"] = np.where(
    agg_features["resource_count"] > 0,
    agg_features["event_count"] / agg_features["resource_count"],
    0.0,
)

agg_features["logs_per_event"] = np.where(
    agg_features["event_count"] > 0,
    agg_features["log_count"] / agg_features["event_count"],
    0.0,
)

agg_features["logvol_per_log"] = np.where(
    agg_features["log_count"] > 0,
    agg_features["log_volume_sum"] / agg_features["log_count"],
    0.0,
)

agg_features["logvol_per_resource"] = np.where(
    agg_features["resource_count"] > 0,
    agg_features["log_volume_sum"] / agg_features["resource_count"],
    0.0,
)

# Log transforms help reduce skew in large volume values
agg_features["log1p_log_volume_sum"] = np.log1p(agg_features["log_volume_sum"])
agg_features["log1p_log_volume_mean"] = np.log1p(agg_features["log_volume_mean"])
agg_features["log1p_log_volume_max"] = np.log1p(agg_features["log_volume_max"])

print("agg_features shape:", agg_features.shape)
show_df(agg_features)


# 9. Merge Full Feature Table

print("\nMERGING FULL FEATURE TABLE")

# Merge all sparse matrices and engineered features into one big feature table
full_features = (
    all_ids
    .merge(severity_matrix, on="id", how="left")
    .merge(event_matrix, on="id", how="left")
    .merge(resource_matrix, on="id", how="left")
    .merge(log_presence_matrix, on="id", how="left")
    .merge(log_volume_matrix, on="id", how="left")
    .merge(agg_features.drop(columns=["location_num"]), on="id", how="left")
)

# Fill missing values created by sparse merges
feature_cols = [c for c in full_features.columns if c not in ["id", "location"]]
full_features[feature_cols] = full_features[feature_cols].fillna(0)

print("full_features shape:", full_features.shape)
show_df(full_features.iloc[:, :20])


# 10. Split Back Into Train and Test

print("\nBUILDING FINAL PROCESSED TABLES")

# Merge features back with train labels
train_processed = (
    train_base[["id", "location", "fault_severity"]]
    .merge(full_features.drop(columns=["location"]), on="id", how="left")
)

# Merge features back with test rows
test_processed = (
    test_base[["id", "location"]]
    .merge(full_features.drop(columns=["location"]), on="id", how="left")
)

# Put key columns at the front for readability
train_cols_front = ["id", "location", "fault_severity"]
test_cols_front = ["id", "location"]

train_other_cols = [c for c in train_processed.columns if c not in train_cols_front]
test_other_cols = [c for c in test_processed.columns if c not in test_cols_front]

train_processed = train_processed[train_cols_front + train_other_cols]
test_processed = test_processed[test_cols_front + test_other_cols]

print("train_processed shape:", train_processed.shape)
print("test_processed  shape:", test_processed.shape)

show_df(train_processed.iloc[:, :20])
show_df(test_processed.iloc[:, :20])


# 11. Final Quality Checks

print("\nFINAL QUALITY CHECKS")

# Check missing values
print("Train missing values total:", int(train_processed.isna().sum().sum()))
print("Test missing values total :", int(test_processed.isna().sum().sum()))

# Check duplicate ids
print("Train duplicate ids:", int(train_processed["id"].duplicated().sum()))
print("Test duplicate ids :", int(test_processed["id"].duplicated().sum()))

# Stop notebook if something is wrong
if train_processed.isna().sum().sum() != 0:
    raise ValueError("train_processed still contains missing values.")

if test_processed.isna().sum().sum() != 0:
    raise ValueError("test_processed still contains missing values.")

if train_processed["id"].duplicated().any():
    raise ValueError("Duplicate IDs found in train_processed.")

if test_processed["id"].duplicated().any():
    raise ValueError("Duplicate IDs found in test_processed.")

print("Quality checks passed.")


# 12. Save Only Important Outputs

print("\nSAVING OUTPUTS")

# These two processed files are the main outputs used later
save_csv(train_processed, PROCESSED_PATH / "train_processed.csv", index=False)
save_csv(test_processed, PROCESSED_PATH / "test_processed.csv", index=False)

# Small feature inventory report
feature_inventory = pd.DataFrame(
    {
        "feature_name": [
            c for c in train_processed.columns
            if c not in ["id", "location", "fault_severity"]
        ]
    }
)

feature_inventory["feature_group"] = feature_inventory["feature_name"].apply(detect_group)
save_csv(feature_inventory, REPORTS_PATH / "feature_inventory.csv", index=False)

# Small summary report 
summary = pd.DataFrame(
    {
        "metric": [
            "train_rows",
            "test_rows",
            "train_columns",
            "test_columns",
            "n_model_features_train",
            "n_event_features",
            "n_resource_features",
            "n_severity_features",
            "n_log_presence_features",
            "n_log_volume_features",
        ],
        "value": [
            len(train_processed),
            len(test_processed),
            train_processed.shape[1],
            test_processed.shape[1],
            train_processed.shape[1] - 3,
            event_matrix.shape[1] - 1,
            resource_matrix.shape[1] - 1,
            severity_matrix.shape[1] - 1,
            log_presence_matrix.shape[1] - 1,
            log_volume_matrix.shape[1] - 1,
        ],
    }
)

save_csv(summary, REPORTS_PATH / "preprocessing_summary.csv", index=False)

print("Saved:")
print("-", PROCESSED_PATH / "train_processed.csv")
print("-", PROCESSED_PATH / "test_processed.csv")
print("-", REPORTS_PATH / "feature_inventory.csv")
print("-", REPORTS_PATH / "preprocessing_summary.csv")

print("\nPREPROCESSING COMPLETE")
print("Processed data saved to:", PROCESSED_PATH)
print("Reports saved to       :", REPORTS_PATH)

Python : 3.12.2
Pandas : 2.3.3
NumPy  : 2.3.5
PROJECT_ROOT   : /Users/hasheenadilmidesilva/Desktop/NetGuard
RAW_DATA_PATH  : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/raw
PROCESSED_PATH : /Users/hasheenadilmidesilva/Desktop/NetGuard/data/processed
REPORTS_PATH   : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/preprocessing

Loaded datasets:
train        : (7381, 3)
test         : (11171, 2)
event_type   : (31170, 2)
log_feature  : (58671, 3)
resource_type: (21076, 2)
severity_type: (18552, 2)

STANDARDIZING LABELS
Unique locations      : 1126
Unique event features : 53
Unique resource feats : 10
Unique severity feats : 5
Unique log features   : 386

BUILDING BASE TABLES
Base all_ids shape: (18552, 3)


,id,location,location_num
0,14121,location 118,118
1,9320,location 91,91
2,14394,location 152,152
3,8218,location 931,931
4,14804,location 120,120



BUILDING SPARSE MATRICES
event_matrix shape       : (18552, 54)
resource_matrix shape    : (18552, 11)
severity_matrix shape    : (18552, 6)
log_presence_matrix shape: (18552, 387)
log_volume_matrix shape  : (18552, 387)

CREATING ENGINEERED FEATURES
agg_features shape: (18552, 21)


,id,location_num,event_count,unique_event_types,resource_count,unique_resource_types,severity_type_count,unique_severity_types,log_count,unique_log_features,log_volume_sum,log_volume_mean,log_volume_max,log_volume_std,events_per_resource,logs_per_event,logvol_per_log,logvol_per_resource,log1p_log_volume_sum,log1p_log_volume_mean,log1p_log_volume_max
0,14121,118,2,2,1,1,1,1,2,2,38,19.000000,19,0.000000,2.0,1.00,19.000000,38.0,3.663562,2.995732,2.995732
1,9320,91,2,2,1,1,1,1,2,2,316,158.000000,200,59.396970,2.0,1.00,158.000000,316.0,5.758902,5.068904,5.303305
2,14394,152,2,2,1,1,1,1,2,2,2,1.000000,1,0.000000,2.0,1.00,1.000000,2.0,1.098612,0.693147,0.693147
3,8218,931,2,2,1,1,1,1,3,3,22,7.333333,12,5.686241,2.0,1.50,7.333333,22.0,3.135494,2.120264,2.564949
4,14804,120,4,4,2,2,1,1,9,9,12,1.333333,2,0.500000,2.0,2.25,1.333333,6.0,2.564949,0.847298,1.098612



MERGING FULL FEATURE TABLE
full_features shape: (18552, 862)


,id,location,location_num,severity_type_1,severity_type_2,severity_type_3,severity_type_4,severity_type_5,event_type_1,event_type_10,event_type_11,event_type_12,event_type_13,event_type_14,event_type_15,event_type_17,event_type_18,event_type_19,event_type_2,event_type_20
0,14121,location 118,118,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,9320,location 91,91,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,14394,location 152,152,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,8218,location 931,931,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0
4,14804,location 120,120,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1



BUILDING FINAL PROCESSED TABLES
train_processed shape: (7381, 863)
test_processed  shape: (11171, 862)


,id,location,fault_severity,location_num,severity_type_1,severity_type_2,severity_type_3,severity_type_4,severity_type_5,event_type_1,event_type_10,event_type_11,event_type_12,event_type_13,event_type_14,event_type_15,event_type_17,event_type_18,event_type_19,event_type_2
0,14121,location 118,1,118,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,9320,location 91,0,91,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,14394,location 152,1,152,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,8218,location 931,1,931,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
4,14804,location 120,0,120,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


,id,location,location_num,severity_type_1,severity_type_2,severity_type_3,severity_type_4,severity_type_5,event_type_1,event_type_10,event_type_11,event_type_12,event_type_13,event_type_14,event_type_15,event_type_17,event_type_18,event_type_19,event_type_2,event_type_20
0,11066,location 481,481,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,18000,location 962,962,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0
2,16964,location 491,491,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4795,location 532,532,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
4,3392,location 600,600,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0



FINAL QUALITY CHECKS
Train missing values total: 0
Test missing values total : 0
Train duplicate ids: 0
Test duplicate ids : 0
Quality checks passed.

SAVING OUTPUTS
Saved:
- /Users/hasheenadilmidesilva/Desktop/NetGuard/data/processed/train_processed.csv
- /Users/hasheenadilmidesilva/Desktop/NetGuard/data/processed/test_processed.csv
- /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/preprocessing/feature_inventory.csv
- /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/preprocessing/preprocessing_summary.csv

PREPROCESSING COMPLETE
Processed data saved to: /Users/hasheenadilmidesilva/Desktop/NetGuard/data/processed
Reports saved to       : /Users/hasheenadilmidesilva/Desktop/NetGuard/reports/preprocessing
